# **Ingest sprints.Json File**
1. Read all the files from the sprints folder using spark dataframe reader API
2. Define and enforce schema 
3. Add Metadata Columns
    .Source File
    .Ingestion Time Stamp 
3. Write to bronze delta table

Note: JSON is in Multi Line Format

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### **Step 1 Define and enforce schema (preserve nested structure)**

In [0]:
#Define the Nested Schema 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType, LongType, DoubleType

sprints_schema = StructType(fields=[    
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", LongType(), True),
    StructField("season", LongType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", LongType(), True),    
    StructField("laps", LongType(), True),
    StructField("number", LongType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", LongType(), True),
    StructField("positionText", StringType(), True),    
    StructField("status", StringType(), True)
])


#### **Step-2 Read the JSon file using the dataframe reader API**

In [0]:
sprints_df = (
    spark.read
    .format("json")
#   .option("inferSchema", True)
    .schema(sprints_schema)
    .option("header", True)
    .option("multiLine",True)
    .option('mode','FAILFAST')
    .load(source_file)
)

#### **Step-3. Add Metadata Columns**

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
sprints_final_df.printSchema()

In [0]:
display(sprints_final_df)

####  **Step 3. Write to bronze delta table**

In [0]:
(
    sprints_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
display(sprints_final_df)

In [0]:
%sql
DESCRIBE HISTORY table_name